# Phase 1 — Testing

Verifies the full pipeline end-to-end:
1. Ingest `headphones.json` into ChromaDB via the ingest script
2. Inspect the stored data
3. Test semantic queries
4. Test the full RAG chain

## 0 — Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path(".").resolve().parent
SRC = REPO_ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Repo root:", REPO_ROOT)
print("src exists:", SRC.exists())

## 1 — Ingest

Delete any existing ChromaDB and re-ingest from `data/headphones.json`.

In [ ]:
import shutil

from chatshop.config import settings
from chatshop.data.cleaner import clean_headphones
from chatshop.data.loader import load_json
from chatshop.embeddings.embedder import Embedder
from chatshop.vectorstore.chroma import ChromaStore

chroma_path = REPO_ROOT / settings.chroma_persist_dir
if chroma_path.exists():
    shutil.rmtree(chroma_path)
    print(f"Deleted existing ChromaDB at {chroma_path}")

raw = load_json(REPO_ROOT / "data" / "headphones.json")
products = clean_headphones(raw)
print(f"{len(raw)} raw → {len(products)} clean products")

embedder = Embedder()
vectors = embedder.encode([p.to_document_text() for p in products])

store = ChromaStore()
store.upsert(products, vectors)
print(f"Ingested {store.count()} products into ChromaDB")

## 2 — Inspect ChromaDB

In [ ]:
import json

from chatshop.vectorstore.chroma import ChromaStore

store = ChromaStore()
print(f"Documents in collection: {store.count()}")

raw = store._collection.peek(limit=1)
print("\nMetadata for first document:")
print(json.dumps(raw["metadatas"][0], indent=2))

In [ ]:
import json

raw_peek = store._collection.peek(limit=1)
print(f"Documents in collection: {store.count()}")
print("\nMetadata for first document:")
print(json.dumps(raw_peek["metadatas"][0], indent=2))

In [ ]:
from chatshop.embeddings.embedder import Embedder

embedder = Embedder()


def show_results(products, label=""):
    if label:
        print(f"\n{'─'*55}\n {label}\n{'─'*55}")
    for i, p in enumerate(products, 1):
        price = f"${p.price:.0f}" if p.price else "N/A"
        flags = []
        if p.wireless:
            flags.append("wireless")
        if p.anc:
            flags.append("ANC")
        if p.use_cases:
            flags.append(", ".join(p.use_cases))
        print(f"[{i}] {p.title}")
        print(f"     {price} | {p.type} | {' | '.join(flags)}")

In [ ]:
def show_results(products, label=""):
    if label:
        print(f"\n{'─'*55}\n {label}\n{'─'*55}")
    for i, p in enumerate(products, 1):
        price = f"${p.price:.0f}" if p.price else "N/A"
        flags = []
        if p.wireless:
            flags.append("wireless")
        if p.anc:
            flags.append("ANC")
        if p.use_cases:
            flags.append(", ".join(p.use_cases))
        print(f"[{i}] {p.title}")
        print(f"     {price} | {p.type} | {' | '.join(flags)}")

In [ ]:
# Semantic + metadata filter: ANC, wireless, under $200
where = {
    "$and": [
        {"wireless": {"$eq": True}},
        {"anc": {"$eq": True}},
        {"price": {"$lte": 200}},
        {"price": {"$gt": 0}},
    ]
}
results = store.query(embedder.encode_one("commute noise cancelling"), top_k=5, where=where)
show_results(results, "ANC + wireless + price ≤ $200")

In [ ]:
# Sport / gym — in-ear, wireless
where_sport = {
    "$and": [
        {"wireless": {"$eq": True}},
        {"type": {"$eq": "in-ear"}},
    ]
}
results = store.query(embedder.encode_one("earbuds for running and gym"), top_k=5, where=where_sport)
show_results(results, "in-ear + wireless (sport)")

## 4 — Full RAG Chain

In [ ]:
from chatshop.config import settings
from chatshop.rag.chain import RAGChain
from chatshop.rag.retriever import Retriever

has_key = bool(settings.litellm_api_key or settings.openrouter_api_key)
print(f"Model  : {settings.litellm_model}")
print(f"API key: {'set' if has_key else 'NOT SET — skipping LLM call'}")

In [ ]:
if has_key:
    retriever = Retriever(embedder=embedder, store=store)
    chain = RAGChain(retriever=retriever)

    query = "I need wireless headphones under $150 good for commuting"
    print(f"Query: {query}\n")
    print(chain.run(query))
else:
    print("Add LITELLM_API_KEY or OPENROUTER_API_KEY to .env to run this cell.")